# SHIA-RAG — QASPER Benchmark (Colab, GPU)

**This is the benchmark SHIA-RAG is actually built for.** QASPER is question-answering over **full research papers** —
long documents with a real **section hierarchy** (Introduction, Methods 3.1, Experiments 4, …). SHIA-RAG induces that
hierarchy and retrieves along it with parent-context guarantees, so *here* its structural advantage can show up in the
answer, unlike narrative QuALITY.

Compares **SHIA-RAG** (leaf-anchored precedence retrieval — the improved engine) and **SHIA-RAG(concept)** (original node-scoring engine, ablation)
against **RAPTOR, TreeRAG, GraphRAG, HiChunk, DTE-RAG** — all answered by the **same GPU model** (Qwen2.5-3B-Instruct),
every method capped to the **same word budget**. Reports the standard QASPER **Answer-F1** and average context size.

**Step 0:** Runtime → Change runtime type → **T4 GPU** → Save. Then run cells top to bottom; upload **shia_core_src.zip** at Cell 2.

In [ ]:
# Cell 1 — install (no `datasets` library needed; QASPER is downloaded directly, so no pyarrow/datasets version clashes)
!pip -q install transformers accelerate scikit-learn pydantic numpy

In [ ]:
# Cell 2 — upload your SHIA-RAG code (shia_core_src.zip)
from google.colab import files
import zipfile, os, sys
print("Choose shia_core_src.zip ...")
up = files.upload(); z = list(up)[0]
os.makedirs("shia_core", exist_ok=True)
with zipfile.ZipFile(z) as f: f.extractall("shia_core")
os.makedirs("stubs", exist_ok=True); open("stubs/pymupdf.py","w").write("# stub")
sys.path.insert(0,"stubs"); sys.path.insert(0,"shia_core")
print("SHIA-RAG code ready:", os.listdir("shia_core"))

In [ ]:
# Cell 3 — load QASPER directly from the official Allen AI release (no `datasets` library)
import urllib.request, tarfile, io, json, os
N_Q = 150         # firmed-up sample (was 60) — statistically solid ranking
URL = "https://qasper-dataset.s3.us-west-2.amazonaws.com/qasper-train-dev-v0.3.tgz"
FN  = "qasper-dev-v0.3.json"

if not os.path.exists(FN):
    print("downloading official QASPER release ...")
    data = urllib.request.urlopen(URL, timeout=180).read()
    with tarfile.open(fileobj=io.BytesIO(data)) as t:
        member = next(m for m in t.getmembers() if m.name.endswith("dev-v0.3.json"))
        open(FN, "wb").write(t.extractfile(member).read())
    print("saved", FN)

D = json.load(open(FN))          # {paper_id: {title, abstract, full_text:[{section_name,paragraphs}], qas:[...]}}
items = []
for pid, paper in D.items():
    sections = [("Abstract", [paper["abstract"]])] if paper.get("abstract") else []
    for sec in paper.get("full_text") or []:
        pl = [p for p in (sec.get("paragraphs") or []) if isinstance(p, str) and p.strip()]
        if pl: sections.append((sec.get("section_name") or "Section", pl))
    if not sections: continue
    article = "\n".join(p for _, ps in sections for p in ps)
    if len(article.split()) < 200: continue
    for qa in paper.get("qas") or []:
        golds = []
        for a in qa.get("answers") or []:
            ao = a.get("answer") or {}
            if ao.get("unanswerable"): continue
            spans = ao.get("extractive_spans") or []
            ff = (ao.get("free_form_answer") or "").strip()
            if spans: golds.append(" ".join(spans))
            elif ff:  golds.append(ff)
            elif ao.get("yes_no") is True: golds.append("Yes")
        golds = [g for g in golds if g.strip()]
        if not golds: continue
        items.append({"q": qa.get("question",""), "golds": golds, "sections": sections, "article": article})
        if len(items) >= N_Q: break
    if len(items) >= N_Q: break

npapers = len({id(it["sections"]) for it in items})
print(f"{len(items)} answerable questions from {npapers} papers")
print("example:", items[0]["q"][:70], "| gold:", items[0]["golds"][0][:50],
      "| paper words:", len(items[0]["article"].split()))

In [ ]:
# Cell 4 — GPU reader (same model answers every method; free-form -> Answer-F1)
import torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer
MID="Qwen/Qwen2.5-3B-Instruct"
tok=AutoTokenizer.from_pretrained(MID)
model=AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16, device_map="auto")
def answer_free(question, context):
    prompt=("Answer the question using ONLY the context, as briefly as possible "
            "(a few words or a short phrase). If the context does not contain the answer, "
            "reply exactly: unanswerable.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\nShort answer:")
    msgs=[{"role":"user","content":prompt}]
    text=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=tok(text, return_tensors="pt").to(model.device)
    out=model.generate(**inp, max_new_tokens=48, do_sample=False)
    r=tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return r.splitlines()[0].strip() if r else ""
print("reader OK ->", answer_free("Which city is France's capital?", "France's capital is Paris."))

In [ ]:
# Cell 5 — structured-document retrievers (equal word budget for all)
import sys, re
import logging; logging.getLogger().setLevel(logging.CRITICAL)
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from src.pipeline import SHIARAGPipeline
from src.layer0_data_model.schemas import DocumentMimeType

# ---- fairness knobs (identical for every method) -------------------------
BUDGET     = 512   # WORDS the reader may see — binding, a QASPER paper is ~4-8k words
SHIA_KEEP  = 400   # SHIA-RAG+ voluntarily keeps fewer, most-relevant words (efficiency)
# --------------------------------------------------------------------------

def chunks(article, size=90):
    w = article.split()
    return [" ".join(w[i:i+size]) for i in range(0, len(w), size)] or [article]

def cap(text, n=BUDGET):
    return " ".join(text.split()[:n])

# ============================ BASELINES ===================================
# Identical mechanism to the HotpotQA / QuALITY harness — they chunk the full
# paper text and pick top passages by their defining signal. Equal word budget.

def flat_retrieve(item):
    C = chunks(item["article"]); q = item["q"]
    v = TfidfVectorizer().fit(C+[q])
    sim = cosine_similarity(v.transform([q]), v.transform(C))[0]
    out,u = [],0
    for i in sim.argsort()[::-1]:
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w
    return "\n".join(out)

def raptor_retrieve(item):
    C = chunks(item["article"]); q = item["q"]
    v = TfidfVectorizer().fit(C+[q]); Md=v.transform(C).toarray(); qv=v.transform([q]).toarray()[0]
    try:
        km=KMeans(n_clusters=min(max(2,int(len(C)**0.5)),len(C)),n_init=5,random_state=0).fit(Md)
        lab=list(km.labels_); ce=km.cluster_centers_
    except Exception:
        lab=[0]*len(C); ce=np.zeros((1,Md.shape[1]))
    ps=(Md@qv)/((np.linalg.norm(Md,axis=1)*np.linalg.norm(qv))+1e-9)
    cs=(ce@qv)/((np.linalg.norm(ce,axis=1)*np.linalg.norm(qv))+1e-9)
    out,u=[],0
    for i in sorted(range(len(C)),key=lambda i:-(ps[i]+0.3*cs[lab[i]])):
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w
    return "\n".join(out)

def treerag_retrieve(item):   # structural locality: top chunks + their neighbours
    C=chunks(item["article"]); q=item["q"]
    v=TfidfVectorizer().fit(C+[q]); sim=cosine_similarity(v.transform([q]),v.transform(C))[0]
    order=list(sim.argsort()[::-1]); keep=set(); seq=[]
    for i in order:
        for j in (i-1,i,i+1):
            if 0<=j<len(C) and j not in keep: keep.add(j); seq.append(j)
        if sum(len(C[j].split()) for j in seq)>BUDGET: break
    out,u=[],0
    for j in sorted(seq):
        w=len(C[j].split())
        if u+w>BUDGET: break
        out.append(C[j]); u+=w
    return "\n".join(out)

def _ents(t): return set(re.findall(r"\b[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*\b",t))
def graphrag_retrieve(item):
    C=chunks(item["article"]); q=item["q"]; qe=_ents(q); pe=[_ents(c) for c in C]
    sc=[len(qe&e) for e in pe]; order=sorted(range(len(C)),key=lambda i:-sc[i])
    out,u,pk=[],0,set()
    for i in order:
        if sc[i]==0 and pk: break
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w; pk.add(i)
    if pk:
        seed=set().union(*[pe[i] for i in pk])
        for i in order:
            if i in pk: continue
            if pe[i]&seed:
                w=len(C[i].split())
                if u+w>BUDGET: break
                out.append(C[i]); u+=w
    return "\n".join(out) if out else flat_retrieve(item)

def hichunk_retrieve(item):   # sentence leaves + auto-merge
    sents=re.split(r"(?<=[.!?])\s+",item["article"]); q=item["q"]
    sents=[s for s in sents if s.strip()] or [item["article"]]
    v=TfidfVectorizer().fit(sents+[q]); sim=cosine_similarity(v.transform([q]),v.transform(sents))[0]
    out,u=[],0
    for j in sim.argsort()[::-1]:
        w=len(sents[j].split())
        if u+w>BUDGET: break
        out.append(sents[j]); u+=w
    return "\n".join(out)

def dte_retrieve(item):
    C=chunks(item["article"]); q=item["q"]
    v=TfidfVectorizer().fit(C+[q]); Mp=v.transform(C); sem=cosine_similarity(v.transform([q]),Mp)[0]
    qt=set(q.lower().split())
    lex=np.array([len(qt&set(c.lower().split()))/((len(qt|set(c.lower().split())))or 1) for c in C])
    lex=lex/(lex.max()+1e-9); base=0.5*sem+0.5*lex; Md=Mp.toarray()
    sel,u,cand=[],0,list(range(len(C)))
    while cand:
        if not sel: i=max(cand,key=lambda i:base[i])
        else:
            i=max(cand,key=lambda i:0.7*base[i]-0.3*max((Md[i]@Md[j])/((np.linalg.norm(Md[i])*np.linalg.norm(Md[j]))+1e-9) for j in sel))
        w=len(C[i].split())
        if u+w>BUDGET: break
        sel.append(i); u+=w; cand.remove(i)
    return "\n".join(C[i] for i in sel)

# ============================ SHIA-RAG ====================================
# Builds the paper's REAL section hierarchy (section -> paragraph), so the
# precedence-constrained retriever has genuine structure to follow. This is
# exactly the setting SHIA-RAG is designed for (unlike narrative QuALITY).

def _num_prefix(name, fallback):
    m=re.match(r"\s*(\d+(?:\.\d+)*)", name or "")
    return m.group(1) if m else str(fallback)

_PIPE_CACHE={}; _BLOCK_CACHE={}   # papers repeat across their questions -> build once
def _build_shia(item):
    """Return (pipeline, text->section_name map). Cached per paper."""
    key=hash(item["article"])
    if key in _PIPE_CACHE: return _PIPE_CACHE[key]
    p=SHIARAGPipeline(); doc=p.register_document("q",DocumentMimeType.PLAIN,pages=1)
    sec_of={}; o=1
    for si,(sname,paras) in enumerate(item["sections"],1):
        base=_num_prefix(sname, si)
        for pj,para in enumerate(paras,1):
            para=para.strip()
            if not para: continue
            t=para[:600]
            p.add_text_block(doc.doc_id, t, o, section_path=f"{base}.{pj}"); o+=1
            sec_of[t]=sname or f"Section {si}"
    for node,par in p.concept_extractor.extract_from_document(doc,list(p.blocks.values())):
        p.add_concept_node(node,candidate_parents=par)
    p.run_confidence_propagation()
    try: p.run_crosslink_discovery()
    except Exception: pass
    p.validate_invariants()
    _PIPE_CACHE[key]=(p, sec_of)
    return p, sec_of

def _shia_blocks(item, mode):
    """Selected block texts in reading order. mode='leaf' (new leaf-anchored engine)
       or 'concept' (original node-scoring engine). Cached per (paper,query,mode)."""
    ck=(hash(item["article"]), item["q"], mode)
    if ck in _BLOCK_CACHE: return _BLOCK_CACHE[ck]
    p, _ = _build_shia(item)
    if mode=="leaf":
        rr = p.retrieve_leaf_anchored(item["q"], token_budget=BUDGET)
    else:
        rr = p.retrieve(item["q"], token_budget=BUDGET)
    seen, out = set(), []
    for nid in rr.selected_node_ids:
        for bid in getattr(p.nodes[nid], "source_block_ids", []):
            b = p.blocks.get(bid)
            if b and b.text_content not in seen:
                seen.add(b.text_content); out.append(b.text_content)
    _BLOCK_CACHE[ck]=out
    return out

def shia_retrieve(item):
    """SHIA-RAG (leaf-anchored precedence retrieval — method of record)."""
    out,u=[],0
    for t in _shia_blocks(item, "leaf"):
        w=len(t.split())
        if u+w>BUDGET: break
        out.append(t); u+=w
    return "\n".join(out) if out else flat_retrieve(item)

def shia_concept_retrieve(item):
    """SHIA-RAG (concept) — original node-scoring engine, kept as an ablation."""
    out,u=[],0
    for t in _shia_blocks(item, "concept"):
        w=len(t.split())
        if u+w>BUDGET: break
        out.append(t); u+=w
    return "\n".join(out) if out else flat_retrieve(item)

RETR={
    "SHIA-RAG":         shia_retrieve,          # leaf-anchored precedence (method of record)
    "SHIA-RAG(concept)":shia_concept_retrieve,  # ablation: original node-scoring engine
    "RAPTOR":           raptor_retrieve,
    "TreeRAG":          treerag_retrieve,
    "GraphRAG":         graphrag_retrieve,
    "HiChunk":          hichunk_retrieve,
    "DTE-RAG":          dte_retrieve,
}

# ============================ QASPER answer-F1 ============================
def _norm(s):
    s=s.lower()
    s=re.sub(r"\b(a|an|the)\b"," ",s)
    s=re.sub(r"[^a-z0-9 ]"," ",s)
    return " ".join(s.split())

def token_f1(pred, gold):
    pt=_norm(pred).split(); gt=_norm(gold).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    common=Counter(pt)&Counter(gt); ns=sum(common.values())
    if ns==0: return 0.0
    prec=ns/len(pt); rec=ns/len(gt)
    return 2*prec*rec/(prec+rec)

def best_f1(pred, golds):
    return max((token_f1(pred,g) for g in golds), default=0.0)


print("retrievers ready:", list(RETR))

In [ ]:
# Cell 6 — run and print the QASPER comparison (Answer-F1)
import time, json
M=list(RETR); f1sum={m:0.0 for m in M}; cw={m:[] for m in M}; n=0; t0=time.time()
rows=[]
for i,it in enumerate(items,1):
    row={"q":it["q"],"golds":it["golds"]}
    for m in M:
        ctx=cap(RETR[m](it))
        cw[m].append(len(ctx.split()))
        pred=answer_free(it["q"], ctx)
        f=best_f1(pred, it["golds"]); f1sum[m]+=f
        row[m]={"pred":pred,"f1":round(f,3)}
    rows.append(row); n+=1
    if i%5==0: print(f"  {i}/{len(items)} done ({time.time()-t0:.0f}s)")
json.dump({"rows":rows}, open("results_qasper.json","w"), indent=2)
order=sorted(M, key=lambda m:-f1sum[m]/n)
print("\n===== QASPER RESULTS  ·  reader = Qwen2.5-3B-Instruct (GPU)  ·  n =", n, "=====")
print(f"{'Method':<12}{'Answer-F1 %':>13}{'AvgCtxWords':>13}")
for m in order: print(f"{m:<12}{100*f1sum[m]/n:>13.1f}{sum(cw[m])/len(cw[m]):>13.0f}")
print("\nReference (reported in papers): RAPTOR QASPER F1 55.7.")
print("Same reader, same word budget for all — differences are purely retrieval.")